In [1]:
# Load dependencies
import pandas as pd
import numpy as np
import json
import warnings
warnings.filterwarnings("ignore")
import getpass
import re
import boto3
import os
from etl import make_df, concat_dfs

In [2]:
# List all the files in the data dir
os.listdir('data')

['computers.json',
 'csv',
 '.ipynb_checkpoints',
 'automative.json',
 'beauty.json',
 'health_household.json',
 'baby.json',
 'art.json',
 'electronics.json']

In [21]:
# Concatenate dataframes
df = concat_dfs('data/')

# Preview
df

Loaded data from computers.json
Loaded data from automative.json
Loaded data from beauty.json
Loaded data from health_household.json
Loaded data from baby.json
Loaded data from art.json
Loaded data from electronics.json


,name,mean_rating,num_ratings,price,division,department
0,"USB 3.0 Hub, INTPW 4 Port USB 3.0 Hub Multipor...",4.5,56,8.39,Computer Accessories & Peripherals,computers
1,"Imegny RGB Gaming Mouse Pad,900x400mm XXXL Ext...",4.8,41,20.99,Computer Accessories & Peripherals,computers
2,Eazy2hD 15U Open Frame Network Rack for Server...,3.9,3,129.99,Computer Accessories & Peripherals,computers
3,"Elebase USB to USB C Adapter 4 Pack,Type C Fem...",4.6,"32,687",9.99,Computer Accessories & Peripherals,computers
4,HP 63XL Black High-yield Ink Cartridge | Works...,4.7,"49,778",45.89,Computer Accessories & Peripherals,computers
...,...,...,...,...,...,...
365160,"Ebook Reader, 7inch TFT LCD E-book Reader, Dig...",2.7,4,66.14,eBook Readers & Accessories,electronics
365161,TiMOVO Sleeve Case for 6.8 inch All-New Kindle...,5.0,2,13.99,eBook Readers & Accessories,electronics
365162,kwmobile Cover Compatible with Kobo Libra H2O ...,5.0,3,12.99,eBook Readers & Accessories,electronics
365163,BoxWave Cable Compatible with Amazon Kindle Oa...,5.0,1,14.95,eBook Readers & Accessories,electronics


In [22]:
# Get the meta data
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 365165 entries, 0 to 365164
Data columns (total 6 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   name         365165 non-null  object
 1   mean_rating  365165 non-null  object
 2   num_ratings  365165 non-null  object
 3   price        365165 non-null  object
 4   division     365165 non-null  object
 5   department   365165 non-null  object
dtypes: object(6)
memory usage: 16.7+ MB


In [23]:
# Type casting columns to numeric
df['price'] = df['price'].str.replace(",", "").astype('float')
df['mean_rating'] = df['mean_rating'].astype('float')
df['num_ratings'] = df['num_ratings'].str.replace(",", "").astype(int)

# preview
df.tail()

,name,mean_rating,num_ratings,price,division,department
365160,"Ebook Reader, 7inch TFT LCD E-book Reader, Dig...",2.7,4,66.14,eBook Readers & Accessories,electronics
365161,TiMOVO Sleeve Case for 6.8 inch All-New Kindle...,5.0,2,13.99,eBook Readers & Accessories,electronics
365162,kwmobile Cover Compatible with Kobo Libra H2O ...,5.0,3,12.99,eBook Readers & Accessories,electronics
365163,BoxWave Cable Compatible with Amazon Kindle Oa...,5.0,1,14.95,eBook Readers & Accessories,electronics
365164,TiMOVO [3 Pack Anti-Glare Screen Protector Des...,2.0,2,10.99,eBook Readers & Accessories,electronics


In [24]:
# duplicates
df['name'].duplicated().sum()

127307

In [26]:
df.shape[0] - df['name'].duplicated().sum()

237858

In [27]:
# Drop duplicates
df = df.drop_duplicates(subset='name', keep=False)
df

,name,mean_rating,num_ratings,price,division,department
3,"Elebase USB to USB C Adapter 4 Pack,Type C Fem...",4.6,32687,9.99,Computer Accessories & Peripherals,computers
4,HP 63XL Black High-yield Ink Cartridge | Works...,4.7,49778,45.89,Computer Accessories & Peripherals,computers
5,"etguuds 2-Pack 3ft USB C Cable 3A Fast Charge,...",4.6,46968,7.99,Computer Accessories & Peripherals,computers
6,"Anker USB C Charger Cable [2 Pack, 6ft], 310 T...",4.7,3981,9.99,Computer Accessories & Peripherals,computers
7,HP 63 Black Ink Cartridge | Works with HP Desk...,4.7,87473,22.89,Computer Accessories & Peripherals,computers
...,...,...,...,...,...,...
365155,BoxWave Case Compatible with Pocketbook Touch ...,5.0,1,25.95,eBook Readers & Accessories,electronics
365156,PocketBook Cover for InkPad X | Black | PU Lea...,3.9,7,21.13,eBook Readers & Accessories,electronics
365157,kwmobile Case Compatible with Kobo Libra H2O -...,4.7,5,12.99,eBook Readers & Accessories,electronics
365158,kwmobile Case Compatible with Kobo Libra 2 Cas...,4.1,13,11.99,eBook Readers & Accessories,electronics


In [20]:
df

,name,mean_rating,num_ratings,price,division,department


In [17]:
def export_data_to_s3(data):
    # s3 client
    s3 = boto3.client('s3')
    csv_data = data.to_csv(index=False)

    bucket_name = "amazon-scraped-products"
    file_name = "auto.csv"

    s3.put_object(Body=csv_data, Bucket=bucket_name, Key=file_name)

    print("Dataframe is saved as CSV in S3 bucket.")
    
export_data_to_s3()

Dataframe is saved as CSV in S3 bucket.
